In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from pydantic import BaseModel, Field

In [4]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.5 
)

In [15]:
# LLM structured response type
class LLMFeedback(BaseModel):
    # Field(..., ...) means 'text' is strictly required
    text: str = Field(..., description="The textual feedback or response.")
    score: int = Field(..., ge=1, le=10, description="Integer score ranging from 1 to 10.")


structured_llm = llm.with_structured_output(LLMFeedback)
essay = """
Yemen, nestled at the southern tip of the Arabian Peninsula, holds a storied place in history as one of the earliest centers of human civilization. Often overlooked in favor of ancient Egypt or Mesopotamia, Yemen’s fertile highlands, strategic maritime position, and complex irrigation systems allowed remarkably sophisticated societies to flourish thousands of years ago. Known to the ancient Romans as *Arabia Felix* ("Happy Arabia"), this land transformed harsh desert edges into thriving oases of wealth, culture, and power.

At the heart of ancient Yemen’s rise was its control over the lucrative Frankincense Trail. Kingdoms such as Saba, Qataban, Hadramawt, and Ma'in amassed vast wealth by harvesting and trading frankincense, myrrh, and rare spices to empires across the Mediterranean and Asia. The legendary Kingdom of Saba—frequently associated with the biblical Queen of Sheba—became an economic powerhouse. Beyond commerce, the Sabaeans were master engineers. Their greatest achievement, the Great Dam of Ma'rib, stood for over a millennium as an engineering marvel, capturing seasonal monsoon rains to irrigate vast swathes of farmland and sustain entire cities in the desert.

The architectural and cultural achievements of ancient Yemen were equally extraordinary. Early Yemenis built towering multi-story structures from stone and mudbrick, developing urban architectural styles that influenced the region for centuries. They constructed intricate temples, fortresses, and carved elaborate inscriptions in the ancient South Arabian script (*Musnad*), documenting their royal lineages, laws, and religious practices. Later, under the Himyarite Kingdom, Yemen unified much of Arabia, shifting religious landscapes and establishing powerful regional dominance before the arrival of Islam in the seventh century.

Today, ancient Yemen leaves behind a extraordinary cultural legacy, preserved in UNESCO World Heritage sites like Old Sana'a and the archaeological ruins of Ma'rib. Though modern conflicts threaten many of these irreplaceable artifacts, Yemen remains an indispensable cradle of human history. Understanding its ancient kingdoms not only illuminates the roots of Arabian civilization, but also reminds us of the ingenuity and resilience of human societies that made the desert bloom.
"""

res = structured_llm.invoke(f"Evaluate the essay based on yamen: {essay} keep the feedback under 40 words ")

In [18]:
res.score

9

In [9]:
class FeedbackDetail(TypedDict):
    text: str
    score: int

class EssayEvalState(TypedDict):
    title: str
    essay: str
    clarity_feedback: FeedbackDetail
    depth_of_analysis_feedback: FeedbackDetail
    language_feedback: FeedbackDetail

    summary: FeedbackDetail

In [19]:
def get_clarity_feedback(state: EssayEvalState): 
    title = state['title']
    essay = state['essay']

    prompt = f"We have a essay written on title {title}. Evaluate the essay based on the clarity of essay. keep the feedback concise and under 60 words. The essay is in ''' delimeter. '''{essay}'''"

    feedback = structured_llm.invoke(prompt)

    return {"clarity_feedback": feedback} 

def get_analysis_feedback(state: EssayEvalState): 
    title = state['title']
    essay = state['essay']

    prompt = f"We have a essay written on title {title}. Evaluate the essay based on the dept of analysis of essay. keep the feedback concise and under 60 words. The essay is in ''' delimeter. '''{essay}'''"

    feedback = structured_llm.invoke(prompt)

    return {"depth_of_analysis_feedback": feedback} 


def get_language_feedback(state: EssayEvalState): 
    title = state['title']
    essay = state['essay']

    prompt = f"We have a essay written on title {title}. Evaluate the essay based on the language of essay. keep the feedback concise and under 60 words. The essay is in ''' delimeter. '''{essay}'''"

    feedback = structured_llm.invoke(prompt)

    return {"language_feedback": feedback} 

In [20]:
def get_eval_summary(state: EssayEvalState): 
    title = state['title']
    clarity_feedback = state['clarity_feedback']
    depth_of_analysis_feedback = state['depth_of_analysis_feedback']
    language_feedback = state['language_feedback']

    prompt = f"We have a essay written on title {title} and also the feedback based on clarity, depth of analysis and language of essay. Summarize the feedbacks and give a final overall feedback of the essay based on clarity, analysis and language. keep the feedback concise and under 60 words. The feedbacks are in ''' delimeters. Clarity feedback: '''{clarity_feedback}''' \n Depth of Analysis feedback: '''{depth_of_analysis_feedback}''' \n Language feedback: '''{language_feedback}''' "

    feedback = structured_llm.invoke(prompt)

    return {"summary": feedback} 

In [33]:
graph = StateGraph(EssayEvalState)

# nodes
graph.add_node('clarity_feedback', get_clarity_feedback)
graph.add_node('analysis_feedback', get_analysis_feedback)
graph.add_node('language_feedback', get_language_feedback)

graph.add_node('summary', get_eval_summary)

# edges
graph.add_edge(START, 'clarity_feedback')
graph.add_edge(START, 'analysis_feedback')
graph.add_edge(START, 'language_feedback')

graph.add_edge('clarity_feedback', 'summary')
graph.add_edge('analysis_feedback', 'summary')
graph.add_edge('language_feedback', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()



In [22]:
initial_state = {
    'title': "Yemen: The Cradle of Ancient Civilizations",
    'essay': """
            Yemen, nestled at the southern tip of the Arabian Peninsula, holds a storied place in history as one of the earliest centers of human civilization. Often overlooked in favor of ancient Egypt or Mesopotamia, Yemen’s fertile highlands, strategic maritime position, and complex irrigation systems allowed remarkably sophisticated societies to flourish thousands of years ago. Known to the ancient Romans as *Arabia Felix* ("Happy Arabia"), this land transformed harsh desert edges into thriving oases of wealth, culture, and power.

At the heart of ancient Yemen’s rise was its control over the lucrative Frankincense Trail. Kingdoms such as Saba, Qataban, Hadramawt, and Ma'in amassed vast wealth by harvesting and trading frankincense, myrrh, and rare spices to empires across the Mediterranean and Asia. The legendary Kingdom of Saba—frequently associated with the biblical Queen of Sheba—became an economic powerhouse. Beyond commerce, the Sabaeans were master engineers. Their greatest achievement, the Great Dam of Ma'rib, stood for over a millennium as an engineering marvel, capturing seasonal monsoon rains to irrigate vast swathes of farmland and sustain entire cities in the desert.

The architectural and cultural achievements of ancient Yemen were equally extraordinary. Early Yemenis built towering multi-story structures from stone and mudbrick, developing urban architectural styles that influenced the region for centuries. They constructed intricate temples, fortresses, and carved elaborate inscriptions in the ancient South Arabian script (*Musnad*), documenting their royal lineages, laws, and religious practices. Later, under the Himyarite Kingdom, Yemen unified much of Arabia, shifting religious landscapes and establishing powerful regional dominance before the arrival of Islam in the seventh century.

Today, ancient Yemen leaves behind a extraordinary cultural legacy, preserved in UNESCO World Heritage sites like Old Sana'a and the archaeological ruins of Ma'rib. Though modern conflicts threaten many of these irreplaceable artifacts, Yemen remains an indispensable cradle of human history. Understanding its ancient kingdoms not only illuminates the roots of Arabian civilization, but also reminds us of the ingenuity and resilience of human societies that made the desert bloom.
     """
}

final_state = workflow.invoke(initial_state)

In [32]:
final_state['summary'].text

# final_state['summary'].score

"The essay excels in clarity and language, offering a well-structured and engaging narrative with sophisticated prose. It provides a solid overview of Yemen's ancient civilizations, demonstrating good analytical depth for its length."